# 5.3 Linear Regression from Scratch

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.03_linear-regression-from-scratch.ipynb)

Companion code for Section 5.3 of the book. Run the cells in order from top to bottom.

**Example 5.3.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

class LinearRegressionScratch:
    '''
    Linear Regression implemented from scratch.
    Supports both OLS (closed-form) and Gradient Descent.
    '''

    def __init__(self, method='ols', learning_rate=0.01, n_epochs=1000,
                 tol=1e-6, verbose=False):
        self.method        = method          # 'ols' or 'gd'
        self.lr            = learning_rate
        self.n_epochs      = n_epochs
        self.tol           = tol             # Convergence tolerance
        self.verbose       = verbose
        self.weights       = None            # β₁,...,βₙ
        self.bias          = None            # β₀
        self.loss_history  = []

    def _add_bias_column(self, X):
        '''Add a column of ones for the bias term.'''
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def fit_ols(self, X, y):
        '''
        Closed-form OLS solution using the Normal Equation:
            β = (XᵀX)⁻¹ Xᵀy
        '''
        X_b  = self._add_bias_column(X)   # Add bias column → X̃
        # Compute (X̃ᵀX̃)⁻¹ X̃ᵀy
        # Use np.linalg.lstsq for numerical stability (handles near-singular matrices)
        beta, _, _, _ = np.linalg.lstsq(X_b, y, rcond=None)
        self.bias    = beta[0]
        self.weights = beta[1:]
        return self

    def fit_gd(self, X, y):
        '''
        Gradient Descent implementation:
            ∂L/∂w = (2/n) Xᵀ(Xw + b - y)
            ∂L/∂b = (2/n) Σ(ŷᵢ - yᵢ)
        '''
        n_samples, n_features = X.shape
        self.weights  = np.zeros(n_features)
        self.bias     = 0.0
        prev_loss     = np.inf

        for epoch in range(self.n_epochs):
            # Forward pass
            y_pred = self.predict(X)
            errors = y_pred - y

            # Compute gradients
            dw = (2 / n_samples) * X.T @ errors      # ∂MSE/∂w
            db = (2 / n_samples) * errors.sum()      # ∂MSE/∂b

            # Update parameters (gradient descent step)
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

            # Track loss
            loss = np.mean(errors**2)
            self.loss_history.append(loss)

            # Verbose logging
            if self.verbose and (epoch+1) % 100 == 0:
                print(f'  Epoch {epoch+1:4d}: MSE = {loss:.6f}')

            # Early stopping on convergence
            if abs(prev_loss - loss) < self.tol:
                print(f'  Converged at epoch {epoch+1}')
                break
            prev_loss = loss

        return self

    def fit(self, X, y):
        if self.method == 'ols':
            return self.fit_ols(X, y)
        elif self.method == 'gd':
            return self.fit_gd(X, y)
        else:
            raise ValueError(f'Unknown method: {self.method}')

    def predict(self, X):
        if self.weights is None:
            raise ValueError('Model not fitted. Call fit() first.')
        return X @ self.weights + self.bias

    def score(self, X, y):
        '''R² score.'''
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - y.mean())**2)
        return 1 - ss_res / ss_tot

    def mse(self, X, y):
        return np.mean((y - self.predict(X))**2)

    def rmse(self, X, y):
        return np.sqrt(self.mse(X, y))

    def __repr__(self):
        if self.weights is not None:
            w_str = ', '.join([f'{w:.4f}' for w in self.weights])
            return f'LinearRegression(bias={self.bias:.4f}, weights=[{w_str}])'
        return 'LinearRegression(unfitted)'


# ── Test on synthetic dataset ─────────────────────────────────────
np.random.seed(42)
X, y = make_regression(n_samples=500, n_features=5, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalise features for gradient descent stability
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Method 1: OLS (closed-form)
model_ols = LinearRegressionScratch(method='ols')
model_ols.fit(X_train_s, y_train)

# Method 2: Gradient Descent
model_gd = LinearRegressionScratch(method='gd', learning_rate=0.1,
                                    n_epochs=1000, verbose=False)
model_gd.fit(X_train_s, y_train)

# Compare results
print('=== Method Comparison ===')
print(f'{'Method':<12} {'Train R²':>10} {'Test R²':>10} {'Test RMSE':>12} {'Weights'}')
print('-'*70)
for name, model in [('OLS', model_ols), ('GradDescent', model_gd)]:
    tr2 = model.score(X_train_s, y_train)
    te2 = model.score(X_test_s, y_test)
    rmse = model.rmse(X_test_s, y_test)
    w = ', '.join([f'{w:.2f}' for w in model.weights])
    print(f'{name:<12} {tr2:>10.6f} {te2:>10.6f} {rmse:>12.4f}  [{w}]')

# Compare with sklearn
from sklearn.linear_model import LinearRegression
sk_model = LinearRegression()
sk_model.fit(X_train_s, y_train)
sk_r2   = sk_model.score(X_test_s, y_test)
sk_rmse = np.sqrt(mean_squared_error(y_test, sk_model.predict(X_test_s)))
print(f'{'Sklearn':<12} {sk_model.score(X_train_s,y_train):>10.6f} {sk_r2:>10.6f} {sk_rmse:>12.4f}  {[round(w,2) for w
in sk_model.coef_]}')

# ── Visualise gradient descent convergence ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax = axes[0]
ax.plot(model_gd.loss_history, 'b-', linewidth=2, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Gradient Descent Convergence', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.grid(alpha=0.3)
ax.axhline(y=model_ols.mse(X_train_s, y_train), color='red', linestyle='--', label=f'OLS optimum')
ax.legend()

# Predictions comparison
ax = axes[1]
y_pred_ols = model_ols.predict(X_test_s)
ax.scatter(y_test, y_pred_ols, alpha=0.6, color='#1976D2', s=30, label='OLS predictions')
lims = [min(y_test.min(), y_pred_ols.min()), max(y_test.max(), y_pred_ols.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual y', fontsize=12)
ax.set_ylabel('Predicted ŷ', fontsize=12)
ax.set_title(f'Actual vs Predicted (R²={model_ols.score(X_test_s,y_test):.4f})',
            fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lr_from_scratch.png', dpi=150, bbox_inches='tight')
plt.show()